# Overnight Glucose (12am–6am)

Reads `ai_readi/preprocessed/cgm.parquet`, checks what timezone the timestamps are in, and computes average glucose during the overnight window (12:00am–6:00am) using **local** time so the window lines up with each participant's actual night, not UTC.

In [1]:
import pandas as pd

df = pd.read_parquet('ai_readi/preprocessed/cgm.parquet')
df.head()

,start_time,end_time,glucose,unit,event_type,device_id,transmitter_id,participant_id,timezone,start_time_local,end_time_local
0,2023-08-10 17:54:10+00:00,2023-08-10 17:54:10+00:00,143.0,mg/dL,EGV,PG15103578,3474CH,1007,America/Los_Angeles,2023-08-10 10:54:10-07:00,2023-08-10 10:54:10-07:00
1,2023-08-10 17:59:10+00:00,2023-08-10 17:59:10+00:00,142.0,mg/dL,EGV,PG15103578,3474CH,1007,America/Los_Angeles,2023-08-10 10:59:10-07:00,2023-08-10 10:59:10-07:00
2,2023-08-10 18:04:10+00:00,2023-08-10 18:04:10+00:00,145.0,mg/dL,EGV,PG15103578,3474CH,1007,America/Los_Angeles,2023-08-10 11:04:10-07:00,2023-08-10 11:04:10-07:00
3,2023-08-10 18:09:10+00:00,2023-08-10 18:09:10+00:00,147.0,mg/dL,EGV,PG15103578,3474CH,1007,America/Los_Angeles,2023-08-10 11:09:10-07:00,2023-08-10 11:09:10-07:00
4,2023-08-10 18:14:10+00:00,2023-08-10 18:14:10+00:00,149.0,mg/dL,EGV,PG15103578,3474CH,1007,America/Los_Angeles,2023-08-10 11:14:10-07:00,2023-08-10 11:14:10-07:00


## What timezone is `start_time` in?

`start_time` / `end_time` are `datetime64[ns, UTC]` — i.e. **universal time**.

The file also ships `start_time_local` / `end_time_local`, already converted per-participant using the `timezone` column below. Use the `_local` columns for anything tied to time-of-day (like an overnight window), since participants span multiple US timezones.

In [2]:
print(df[['start_time', 'start_time_local', 'timezone']].dtypes)
print()
print('Distinct participant timezones:')
print(df['timezone'].value_counts())

start_time                          datetime64[ns, UTC]
start_time_local    datetime64[us, America/Los_Angeles]
timezone                                         object
dtype: object

Distinct participant timezones:
timezone
America/Los_Angeles    4062400
America/Chicago        2131753
Name: count, dtype: int64


## Flag the overnight window (12am–6am local time)

The window is `[00:00, 06:00)` in local time — midnight included, 6:00am excluded.

In [3]:
local_hour = df['start_time_local'].dt.hour
df['is_overnight'] = local_hour < 6  # 0,1,2,3,4,5

df['is_overnight'].mean()  # fraction of readings that fall in the overnight window

np.float64(0.2520370420297981)

## Average overnight glucose per participant

In [4]:
overnight = df[df['is_overnight']]

overnight_avg_by_participant = (
    overnight.groupby('participant_id')['glucose']
    .agg(mean_glucose='mean', n_readings='count')
    .reset_index()
    .sort_values('participant_id')
)
overnight_avg_by_participant

,participant_id,mean_glucose,n_readings
0,1001,121.500000,720
1,1002,131.470486,576
2,1003,199.265278,720
3,1004,171.126389,720
4,1005,302.123457,648
...,...,...,...
2235,7814,99.837963,432
2236,7815,123.930556,432
2237,7816,111.759722,720
2238,7817,130.175953,682


## Overall average overnight glucose (all participants, all nights)

In [5]:
overall_mean = overnight['glucose'].mean()
print(f"Overall mean overnight (12am-6am local) glucose: {overall_mean:.1f} mg/dL over {len(overnight):,} readings")

Overall mean overnight (12am-6am local) glucose: 131.2 mg/dL over 1,561,156 readings


## (Optional) Average overnight glucose per participant, per night

Useful if you want night-to-night variability rather than one number pooling all nights together. The "night" is labeled by the calendar date at the *start* of the window (so 12am–6am on 2023-08-11 is night `2023-08-11`).

In [6]:
overnight = overnight.copy()
overnight['night_date'] = overnight['start_time_local'].dt.date

overnight_by_night = (
    overnight.groupby(['participant_id', 'night_date'])['glucose']
    .agg(mean_glucose='mean', n_readings='count')
    .reset_index()
)
overnight_by_night

,participant_id,night_date,mean_glucose,n_readings
0,1001,2023-07-28,116.888889,72
1,1001,2023-07-29,119.347222,72
2,1001,2023-07-30,114.430556,72
3,1001,2023-07-31,125.333333,72
4,1001,2023-08-01,121.319444,72
...,...,...,...,...
21762,7818,2025-05-07,111.930556,72
21763,7818,2025-05-08,105.000000,72
21764,7818,2025-05-09,107.444444,72
21765,7818,2025-05-10,108.111111,72


## Average overnight glucose per participant, across nights

This takes `overnight_by_night` (one mean per participant per night) and averages *those* nightly means together, so every night counts equally regardless of how many readings it had (unlike the pooled version above, which implicitly weights nights with more readings more heavily).

In [7]:
overnight_avg_of_nightly_means = (
    overnight_by_night.groupby('participant_id')
    .agg(mean_of_nightly_means=('mean_glucose', 'mean'), n_nights=('night_date', 'nunique'))
    .reset_index()
    .sort_values('participant_id')
)
overnight_avg_of_nightly_means

,participant_id,mean_of_nightly_means,n_nights
0,1001,121.500000,10
1,1002,131.470486,10
2,1003,199.265278,10
3,1004,171.126389,10
4,1005,310.985455,10
...,...,...,...
2235,7814,99.837963,6
2236,7815,123.930556,6
2237,7816,111.759722,10
2238,7817,130.337273,10


In [8]:
n_total_participants = df['participant_id'].nunique()
n_with_overnight = overnight_avg_of_nightly_means['participant_id'].nunique()
print(f"Participants with >=1 overnight reading: {n_with_overnight} of {n_total_participants} "
      f"({n_total_participants - n_with_overnight} dropped, no overnight data at all)")
print(overnight_avg_of_nightly_means['n_nights'].describe())

sparse_nights = overnight_by_night[overnight_by_night['n_readings'] < 10]
print(f"\nNights with <10 readings (very sparse): {len(sparse_nights)} of {len(overnight_by_night)} "
      f"({len(sparse_nights) / len(overnight_by_night) * 100:.1f}%) — these are still included, not dropped")

Participants with >=1 overnight reading: 2240 of 2245 (5 dropped, no overnight data at all)
count    2240.000000
mean        9.717411
std         1.065124
min         1.000000
25%        10.000000
50%        10.000000
75%        10.000000
max        13.000000
Name: n_nights, dtype: float64

Nights with <10 readings (very sparse): 65 of 21767 (0.3%) — these are still included, not dropped


## Add mean-of-nightly-means to `final_df.csv`

Merges `mean_of_nightly_means` (overnight glucose, equal-weighted per night) into the existing `final_df.csv` on `participant_id`, and overwrites the file.

In [9]:
final_df = pd.read_csv('final_df.csv')

overnight_merge = overnight_avg_of_nightly_means[['participant_id', 'mean_of_nightly_means']].copy()
overnight_merge['participant_id'] = overnight_merge['participant_id'].astype(int)

final_df = final_df.drop(columns=['mean_of_nightly_means'], errors='ignore')
final_df = final_df.merge(overnight_merge, on='participant_id', how='left')

final_df.to_csv('final_df.csv', index=False)
print(f"Matched {final_df['mean_of_nightly_means'].notna().sum()} of {len(final_df)} participants in final_df.csv")
final_df[['participant_id', 'mean_of_nightly_means']].head()

Matched 2182 of 2217 participants in final_df.csv


,participant_id,mean_of_nightly_means
0,1001,121.500000
1,1002,131.470486
2,1003,199.265278
3,1004,171.126389
4,1005,310.985455


## MAGE (Mean Amplitude of Glycemic Excursions)

MAGE measures the average size of "significant" glucose swings — it ignores small wobbles and only counts excursions bigger than 1 SD.

Method (classic turning-point algorithm, Service et al. 1970), computed **per calendar day** then averaged across days per participant (same equal-weight-per-day approach as `mean_of_nightly_means` above):
1. For each participant-day, take the local glucose series in time order.
2. Find turning points (local peaks/nadirs — where the series changes direction).
3. Compute that day's SD of glucose.
4. Keep only turning-point-to-turning-point swings whose absolute size exceeds 1 SD.
5. MAGE for that day = mean absolute size of the kept swings.
6. A day only counts if it has at least 202 readings (~70% of the expected 288 five-minute samples/day) — the standard CGM data-sufficiency threshold.
7. A participant's final MAGE = mean of their per-day MAGE values.

Note: there's no single universal standard implementation of MAGE across the literature — this is the classic 1-SD turning-point method. If this feeds a publication, worth cross-checking against a validated tool (e.g. the R `iglu` package).

In [10]:
import numpy as np

cgm_full = (
    df[['participant_id', 'start_time_local', 'glucose']]
    .dropna(subset=['glucose'])
    .sort_values(['participant_id', 'start_time_local'])
    .copy()
)
cgm_full['local_day'] = cgm_full['start_time_local'].dt.date

MIN_READINGS_PER_DAY = 202  # ~70% of the expected 288 five-minute samples/day


def mage_for_day(glucose_values):
    g = np.asarray(glucose_values, dtype=float)
    if len(g) < 3:
        return np.nan
    sd = g.std()
    if sd == 0:
        return np.nan

    turning_idx = [0]
    for i in range(1, len(g) - 1):
        if (g[i] - g[i - 1]) * (g[i + 1] - g[i]) < 0:
            turning_idx.append(i)
    turning_idx.append(len(g) - 1)

    tp_vals = g[turning_idx]
    excursions = np.diff(tp_vals)
    excursions = excursions[np.abs(excursions) > sd]
    if len(excursions) == 0:
        return np.nan
    return np.abs(excursions).mean()


daily_mage = (
    cgm_full.groupby(['participant_id', 'local_day'])['glucose']
    .agg(mage=mage_for_day, n_readings='count')
    .reset_index()
)
daily_mage = daily_mage[daily_mage['n_readings'] >= MIN_READINGS_PER_DAY]

mage_by_participant = (
    daily_mage.groupby('participant_id')
    .agg(mage=('mage', 'mean'), n_days=('mage', 'count'))
    .reset_index()
    .sort_values('participant_id')
)
mage_by_participant

,participant_id,mage,n_days
0,1001,36.078507,9
1,1002,28.193838,7
2,1003,81.625750,9
3,1004,53.736287,9
4,1005,82.311111,9
...,...,...,...
2226,7814,32.648237,6
2227,7815,27.078361,6
2228,7816,40.932517,9
2229,7817,43.257050,9


In [11]:
all_daily_counts = cgm_full.groupby(['participant_id', 'local_day']).size()
n_dropped_days = (all_daily_counts < MIN_READINGS_PER_DAY).sum()
print(f"Participant-days dropped for <{MIN_READINGS_PER_DAY} readings: {n_dropped_days} of {len(all_daily_counts)} "
      f"({n_dropped_days / len(all_daily_counts) * 100:.1f}%)")

n_total_participants = df['participant_id'].nunique()
n_with_mage = mage_by_participant['participant_id'].nunique()
n_no_valid_days = n_total_participants - n_with_mage
print(f"Participants with >=1 valid MAGE day: {n_with_mage} of {n_total_participants} "
      f"({n_no_valid_days} had zero valid days -> MAGE = NaN)")
print(mage_by_participant['n_days'].describe())

Participant-days dropped for <202 readings: 4566 of 24010 (19.0%)
Participants with >=1 valid MAGE day: 2231 of 2245 (14 had zero valid days -> MAGE = NaN)
count    2231.000000
mean        8.712685
std         1.051761
min         1.000000
25%         9.000000
50%         9.000000
75%         9.000000
max        12.000000
Name: n_days, dtype: float64


## MODD (Mean Of Daily Differences)

MODD measures day-to-day consistency at the same clock time — e.g. is glucose at 2am roughly the same every night, or wildly different night to night?

Method:
1. For each participant, bin the local glucose series into a fixed 5-minute clock grid (`resample('5min')`), matching the CGM's own ~5-minute sampling cadence. Missing bins become `NaN`.
2. Shift the series by exactly 288 bins (24 hours) and take `|glucose(t) - glucose(t - 24h)|` for every bin that has both a value and a same-clock-time value one day earlier.
3. MODD = mean of those absolute differences across the whole recording period.

In [12]:
MIN_PAIRS = 288  # require at least one full day's worth of matched adjacent-day pairs


def modd_for_group(sub):
    s = sub.set_index('start_time_local')['glucose'].sort_index()
    s = s.resample('5min').mean()
    diffs = (s - s.shift(288)).abs().dropna()
    return pd.Series({'modd': diffs.mean() if len(diffs) > 0 else np.nan, 'n_pairs': len(diffs)})


modd_by_participant = (
    cgm_full.groupby('participant_id')
    .apply(modd_for_group, include_groups=False)
    .reset_index()
)
modd_by_participant.loc[modd_by_participant['n_pairs'] < MIN_PAIRS, 'modd'] = np.nan
modd_by_participant = modd_by_participant.sort_values('participant_id')
modd_by_participant

,participant_id,modd,n_pairs
0,1001,13.694704,2568.0
1,1002,16.250622,2011.0
2,1003,36.166862,2559.0
3,1004,35.678349,2568.0
4,1005,66.542132,2326.0
...,...,...,...
2239,7814,21.051871,1523.0
2240,7815,13.520740,1567.0
2241,7816,21.265704,2563.0
2242,7817,25.661582,2491.0


In [13]:
n_total_participants = df['participant_id'].nunique()
n_zero_pairs = (modd_by_participant['n_pairs'] == 0).sum()
n_below_floor = (modd_by_participant['n_pairs'] < MIN_PAIRS).sum()
n_valid_modd = modd_by_participant['modd'].notna().sum()
print(f"Participants with 0 matched adjacent-day pairs: {n_zero_pairs} of {n_total_participants}")
print(f"Participants below the {MIN_PAIRS}-pair floor (MODD set to NaN): {n_below_floor} of {n_total_participants}")
print(f"Participants with a valid MODD: {n_valid_modd} of {n_total_participants}")
print(modd_by_participant['n_pairs'].describe())

Participants with 0 matched adjacent-day pairs: 7 of 2245
Participants below the 288-pair floor (MODD set to NaN): 13 of 2245
Participants with a valid MODD: 2231 of 2245
count    2244.000000
mean     2443.077986
std       358.002746
min         0.000000
25%      2520.000000
50%      2561.000000
75%      2568.000000
max      3121.000000
Name: n_pairs, dtype: float64


## Add MAGE and MODD to `final_df.csv`

Merges `mage` and `modd` into the existing `final_df.csv` (which already has `mean_of_nightly_means`) on `participant_id`, and overwrites the file.

In [14]:
final_df = pd.read_csv('final_df.csv')

mage_merge = mage_by_participant[['participant_id', 'mage']].copy()
mage_merge['participant_id'] = mage_merge['participant_id'].astype(int)

modd_merge = modd_by_participant[['participant_id', 'modd']].copy()
modd_merge['participant_id'] = modd_merge['participant_id'].astype(int)

final_df = final_df.drop(columns=['mage', 'modd'], errors='ignore')
final_df = final_df.merge(mage_merge, on='participant_id', how='left')
final_df = final_df.merge(modd_merge, on='participant_id', how='left')

final_df.to_csv('final_df.csv', index=False)
print(f"Matched MAGE for {final_df['mage'].notna().sum()} of {len(final_df)} participants")
print(f"Matched MODD for {final_df['modd'].notna().sum()} of {len(final_df)} participants")
final_df[['participant_id', 'mage', 'modd']].head()

Matched MAGE for 2173 of 2217 participants
Matched MODD for 2173 of 2217 participants


,participant_id,mage,modd
0,1001,36.078507,13.694704
1,1002,28.193838,16.250622
2,1003,81.625750,36.166862
3,1004,53.736287,35.678349
4,1005,82.311111,66.542132


## Methods summary

**Data.** CGM readings from `ai_readi/preprocessed/cgm.parquet` (Dexcom G6, ~5-minute sampling cadence, 2,245 participants). Raw `start_time`/`end_time` are UTC; all variables below use the pre-computed `start_time_local`/`end_time_local` columns (participant-specific local time, spanning `America/Los_Angeles` and `America/Chicago`) so that clock-time-based windows and day boundaries line up with each participant's actual local day/night.

**Overnight mean glucose (`mean_of_nightly_means`).** For each participant-night, readings with local hour in `[0, 6)` (12:00am–5:59am) were averaged to give one mean per night; those nightly means were then averaged again per participant, giving every night equal weight regardless of how many readings it contained. **5 of 2,245 participants (0.2%) had no overnight readings at all and are NaN.** Of the 21,767 participant-nights, 65 (0.3%) had fewer than 10 readings that night — these sparse nights were still included (no per-night minimum was applied), so a small number of nightly means are based on very few points. Matched into `final_df.csv` for 2,182 of 2,217 participants.

**MAGE (`mage`) — Mean Amplitude of Glycemic Excursions.** Classic turning-point method (Service et al., 1970), applied per local calendar day: for each participant-day with at least 202 readings (≥70% of the expected 288 five-minute samples — the standard CGM data-sufficiency threshold), turning points (local peaks/nadirs) in the day's glucose curve were identified, and the absolute size of each turning-point-to-turning-point excursion was compared against that day's own SD of glucose. Excursions exceeding 1 SD were averaged to give that day's MAGE. Per-participant MAGE is the mean of its daily MAGE values (one day = one vote). **4,566 of 24,010 participant-days (19.0%) were dropped for having <202 readings. 14 of 2,245 participants (0.6%) had zero valid days and are NaN**; the remaining 2,231 participants averaged ~9 valid days each. Matched into `final_df.csv` for 2,173 of 2,217 participants. Note: MAGE has no single universal computational standard in the literature; this is the classic 1-SD turning-point implementation and should be cross-checked against a validated tool (e.g. R's `iglu` package) before use in a publication.

**MODD (`modd`) — Mean Of Daily Differences.** Each participant's glucose series was binned onto a regular 5-minute local-clock grid (`resample('5min')`); empty bins were left as missing (no interpolation). For every bin with a value exactly 24 hours (288 bins) after another valid bin, the absolute difference between the two was computed. All such pairs across a participant's entire recording period (every adjacent-day boundary they have, not just one day) were pooled and averaged to give one MODD value per participant. Missingness was low overall (median ~0.03% of bins missing per participant) but variable across participants (up to 90% for a few). **7 of 2,245 participants (0.3%) had zero matched adjacent-day pairs. Participants with fewer than 288 valid pairs (less than one full day's worth of matched comparisons) were set to NaN rather than reported from a handful of pairs — 13 of 2,245 participants (0.6%) fell below this floor**, leaving 2,231 with a valid MODD. Matched into `final_df.csv` for 2,173 of 2,217 participants.